In [14]:
import gradio
from notion_client import Client
import langchain, langchain_chroma
from chromadb import PersistentClient
import openai
import litellm
from dotenv import load_dotenv
import frontmatter
import os
from pprint import pprint
from notion_to_md import NotionToMarkdown
from ingest import (
    sync_notion_notes,
    load_markdown_dir,
    chunk_documents,
    rebuild_chroma_index,
)
from utils import get_n2m_client, convert_page_to_md, post_process_md
from pathlib import Path
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from answer import answer_question

load_dotenv()


True

## Set up notion client


In [3]:
# notion = Client(auth=os.environ["NOTION_TOKEN"])
# n2m = NotionToMarkdown(notion)


In [ ]:
# # Export a page as a markdown blocks
# md_blocks = n2m.page_to_markdown("page_id")

# # Convert markdown blocks to string
# md_str = n2m.to_markdown_string(md_blocks).get("parent")

# # Write to a file
# with open("data/notion_markdown/output2.md", "w") as f:
#     f.write(md_str)

In [5]:
sync_notion_notes()

Loaded 7 markdown pages
Created 1677 chunks from 7 pages


'Exported 7/7 pages | Skipped 0 | Failed 0 | Bytes written 531692\nIndexed 1677 chunks from 7 pages into notion_notes in 24.9s'

### Convert notion pages markdown to chunks (this is already being done in sync_notion_notes)


### Just for fun visualisation


In [10]:
chroma = PersistentClient(Path("chroma_db"))
collection = chroma.get_or_create_collection("notion_notes")
result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors = np.array(result["embeddings"])
documents = result["documents"]
metadatas = result["metadatas"]
doc_types = [metadata["title"] for metadata in metadatas]
colors = [
    ["blue", "green", "red", "orange", "yellow", "purple", "cyan"][
        [
            "Playwright crash course by chatgpt",
            "Playwright telegram bot project v1",
            "Week 1 - Building your first LLM Product: Exploring Top Models + how to create python virtual environment (.venv folder) + technical guides, slides, resources",
            "Week 2 - Build a Multi-Modal Chatbot: LLMs, Gradio UI, and Agents + callbacks crashcourse",
            "Week 3 - Open-Source Gen AI: Automated Solutions with HuggingFace + TONS of technical concepts + uncovering what’s behind LLM models",
            "Week 4 - Evaluating models for Code Gen & Business Tasks + how to delete merged branches from github and locally + groq and openrouter + how to collapse a cell in ipynb in vscode",
            "Week 5 - Mastering RAG: Build Advanced Solutions with Vector Embeddings",
        ].index(t)
    ]
    for t in doc_types
]


In [12]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(
    data=[
        go.Scatter(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            mode="markers",
            marker=dict(size=5, color=colors, opacity=0.8),
            text=[
                f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)
            ],
            hoverinfo="text",
        )
    ]
)

fig.update_layout(
    title="2D Chroma Vector Store Visualization",
    scene=dict(xaxis_title="x", yaxis_title="y"),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40),
)

fig.show()


In [13]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            z=reduced_vectors[:, 2],
            mode="markers",
            marker=dict(size=5, color=colors, opacity=0.8),
            text=[
                f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)
            ],
            hoverinfo="text",
        )
    ]
)

fig.update_layout(
    title="3D Chroma Vector Store Visualization",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40),
)

fig.show()


In [ ]:
answer_question("Have I covered advanced RAG techniques in my notes before?")